# Where do the errors come from?

Two models were given **exactly the same blocks** — the same cached extraction, the
same text, in the same order. One scored 0.30, the other 0.69.

This notebook explains that gap. The short version: pdfplumber cuts a paragraph into
several blocks, and whether that becomes an error depends entirely on whether the
model labels the pieces **consistently**.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    _confusion_from_match, _match_structured, containment, extract_gold,
    load_method, micro_prf1, resolve_old_gt_path, tokenize,
)

MODELS    = ['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b']
EXTRACTOR = 'pdfplumber'
SAMPLES   = range(1, 11)
EXAMPLE   = 5          # the document used for the worked example in section 3

pd.set_option('display.max_colwidth', 62)

INK, MUTED, SURFACE, GRID = '#0b0b0b', '#52514e', '#fcfcfb', '#e6e6e2'
LABEL_TINT = {
    'title':               '#e3edfa',
    'section.title':       '#fce8df',
    'section.description': '#ddf3ec',
    'question.text':       '#ece3fa',
    'answer.text':         '#eceef0',
}
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.edgecolor': '#d8d8d4', 'axes.titlecolor': INK,
    'font.size': 10, 'axes.titlesize': 11.5, 'axes.titleweight': 'bold',
    'axes.grid': False, 'legend.frameon': False,
})


def tint_label(v):
    c = LABEL_TINT.get(v)
    return f'background-color: {c}; color: {INK}' if c else ''


available = [m for m in MODELS
             if all(P.structured_path(P.make_tag(m, EXTRACTOR), n).exists()
                    for n in SAMPLES)]
print('models with a complete run:', ', '.join(available) or 'none')
for m in MODELS:
    if m not in available:
        print('  not yet run, skipped:', m)


models with a complete run: llama3.1:8b, gemma4:e4b
  not yet run, skipped: llama3.3:70b


## 1. The PDF is cut into more pieces than the annotation has items

`pdfplumber` reads **line by line**. A paragraph that wraps across six lines becomes
six blocks. The annotation records that paragraph as **one** item.

This happens before any model runs, and is identical for every model.


In [2]:
rows = []
for n in SAMPLES:
    gold = extract_gold(resolve_old_gt_path(n))
    blocks = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'sample{n}.json')
                        .read_text(encoding='utf-8'))
    pieces = {}
    for b in blocks:
        btok = tokenize(b['text'])
        best, bi = 0.0, None
        for gi, (gt, _) in enumerate(gold):
            c = containment(btok, tokenize(gt))
            if c > best:
                best, bi = c, gi
        if best >= 0.75:
            pieces[bi] = pieces.get(bi, 0) + 1
    rows.append({'sample': f'sample{n}', 'items in annotation': len(gold),
                 'blocks from pdfplumber': len(blocks),
                 'worst single split': max(pieces.values()) if pieces else 0})

frag = pd.DataFrame(rows).set_index('sample')
frag.loc['TOTAL'] = [frag['items in annotation'].sum(),
                     frag['blocks from pdfplumber'].sum(),
                     frag['worst single split'].max()]
display(frag.style.background_gradient(cmap='Oranges',
                                       subset=['worst single split']))


,items in annotation,blocks from pdfplumber,worst single split
sample,,,
sample1,28,78,19
sample2,25,171,30
sample3,14,69,18
sample4,2,78,77
sample5,13,80,23
sample6,11,24,6
sample7,2,17,16
sample8,13,59,17
sample9,11,85,32


**This is not damage.** No text is lost and none is invented — the extractor reads
the page correctly. It simply cuts in different places than the annotation does.

Whether that costs anything depends on what happens next.


## 2. The pipeline can put the pieces back together

After labelling, `to_structured()` **merges consecutive blocks that share a label**.

So six fragments of one answer, all labelled `answer.text`, are rejoined into a
single item automatically — the fragmentation costs nothing.

But the merge only fires on *consecutive, identical* labels. Change the label
part-way through the paragraph and the pieces can never rejoin:

```
answer  answer  answer  answer   ->  1 item     matches the annotation
answer  question  answer  question  ->  4 items  1 matches, 3 are wrong
```

**This is the whole story.** Fragmentation creates the opportunity for an error;
only inconsistent labelling turns it into one.


## 3. The same blocks, two models

Below is one stretch of a real document. The `text` column is identical for both
models — it is the same file. Only the labels differ.


In [3]:
labeled = {m: json.loads(P.labeled_path(P.make_tag(m, EXTRACTOR), EXAMPLE)
                         .read_text(encoding='utf-8')) for m in available}
n_blocks = len(next(iter(labeled.values())))

# Find the run of blocks where the models disagree most — that is the
# interesting part of the document, and it is different for each pair.
if len(available) >= 2:
    a, b = available[0], available[1]
    disagree = [i for i in range(n_blocks)
                if labeled[a][i].get('label') != labeled[b][i].get('label')]
    start = max(0, (disagree[len(disagree) // 2] if disagree else 0) - 5)
else:
    start = 0
window = range(start, min(start + 14, n_blocks))

ex = pd.DataFrame([{**{m: labeled[m][i].get('label') for m in available},
                    'text': labeled[available[0]][i]['text']} for i in window],
                  index=[f'block {i}' for i in window])
print(f'sample{EXAMPLE}, blocks {window.start}-{window.stop - 1} '
      f'— the same text given to every model')
display(ex.style.map(tint_label, subset=available))


sample5, blocks 36-49 — the same text given to every model


,llama3.1:8b,gemma4:e4b,text
block 36,answer.text,question.text,correct attribution to original collectors. All methodological procedures for data quality control and
block 37,question.text,answer.text,aggregation will be described in the data documentation. This information will be provided in the form of
block 38,answer.text,question.text,"supplementary documentation for publications, in the README.txt file, and at the repository (record-"
block 39,answer.text,answer.text,"level). All sharable raw data and the processed data will be archived in Dryad, https://datadryad.org, the"
block 40,section.description,answer.text,"agnostic open access repository backed by UCSB and the California Digital Library (CDL), which"
block 41,question.text,answer.text,"implements the latest best practices in data curation, publication, citation, and archival. Dryad assigns"
block 42,answer.text,answer.text,Digital Object Identifiers (DOIs) to deposited datasets and makes data discoverable through such services
block 43,question.text,answer.text,"as the Thomson-Reuters Data Citation Index, Scopus, and Google Dataset Search. All data in Dryad is"
block 44,answer.text,answer.text,publicly accessible and released under a CC0 license. Data will be linked to publications and vice versa
block 45,question.text,answer.text,through DOIs.


Read a run of rows where the `text` column is clearly one continuous paragraph.

Where one model holds a single colour down the run, those blocks **merge into one
item** and score as one correct answer. Where the colour alternates, the same text
becomes several items — one matches the annotation and the rest are counted as
wrong, even though every one of them is correct text.


## 4. How often each model changes its mind

A **run** is a stretch of consecutive blocks carrying the same label. Fewer runs
means longer stretches, which means more fragments get rejoined.

With one block per line, a model that never changed label would produce very few
runs; one that changed on every block would produce as many runs as there are
blocks.


In [4]:
def label_runs(blocks):
    labs = [b.get('label') for b in blocks]
    return 1 + sum(1 for i in range(1, len(labs)) if labs[i] != labs[i - 1])


rows = []
for m in available:
    tag = P.make_tag(m, EXTRACTOR)
    blocks = runs_ = 0
    for n in SAMPLES:
        bl = json.loads(P.labeled_path(tag, n).read_text(encoding='utf-8'))
        blocks += len(bl)
        runs_ += label_runs(bl)
    f1 = micro_prf1(load_method(tag, exclude=[])[1])['f1']
    rows.append({'model': m, 'blocks': blocks, 'label runs': runs_,
                 'runs per block': runs_ / blocks, 'f1': f1})

runs_df = pd.DataFrame(rows).set_index('model')
display(runs_df.style
        .background_gradient(cmap='Reds', subset=['runs per block'])
        .background_gradient(cmap='Greens', subset=['f1'])
        .format({'runs per block': '{:.2f}', 'f1': '{:.3f}'}))


,blocks,label runs,runs per block,f1
model,,,,
llama3.1:8b,729,461,0.63,0.298
gemma4:e4b,729,164,0.22,0.689


**`runs per block` is the number to watch.** A value near 1.0 means the model changes
label on almost every block — it is deciding each line in isolation, with no memory of
the line before. A lower value means it is carrying context across the break.

Compare that column against `f1`. They move together, and that is the finding: on
line-level input, **label consistency predicts the score better than anything else**.


## 5. Errors per document

Every wrong item falls into one of three kinds:

| | what it means |
|---|---|
| **split apart** | real text, but this item was already counted from another piece of itself — the model broke the paragraph up |
| **wrong label** | the right text, given the wrong one of the five labels |
| **fused block** | one block holding two annotation items at once, so it matches neither — the extractor could not separate them |

Only the last is purely the extractor's: it happens when a heading and its answer
share a line, and nothing in the block signals the boundary. Everything else depends
on how the model labelled what it was given.


In [5]:
def attribute(model):
    tag = P.make_tag(model, EXTRACTOR)
    out = []
    for n in SAMPLES:
        gold = extract_gold(resolve_old_gt_path(n))
        rec, nogold = _match_structured(P.structured_path(tag, n), gold)
        m = micro_prf1(_confusion_from_match(rec, nogold))
        split = fused = 0
        for text, _ in nogold:
            t = tokenize(text)
            if any(containment(t, tokenize(gt)) >= 0.75 for gt, _ in gold):
                split += 1
            else:
                fused += 1
        wrong = sum(1 for r in rec if r['pred_label']
                    and r['pred_label'] != r['gold_label'])
        out.append({'sample': f'sample{n}', 'f1': m['f1'], 'split apart': split,
                    'wrong label': wrong, 'fused block': fused})
    return pd.DataFrame(out).set_index('sample')


for m in available:
    d = attribute(m)
    d.loc['TOTAL'] = d.sum()
    d.loc['TOTAL', 'f1'] = np.nan
    print(m)
    display(d.style
            .background_gradient(cmap='Oranges', subset=['split apart'])
            .background_gradient(cmap='Blues', subset=['wrong label'])
            .format({'f1': '{:.3f}', 'split apart': '{:.0f}',
                     'wrong label': '{:.0f}', 'fused block': '{:.0f}'},
                    na_rep=''))


llama3.1:8b


,f1,split apart,wrong label,fused block
sample,,,,
sample1,0.644,3,9,0
sample2,0.361,13,11,1
sample3,0.193,55,6,0
sample4,0.087,42,0,0
sample5,0.244,64,2,0
sample6,0.057,15,5,3
sample7,0.400,6,0,0
sample8,0.571,16,1,0
sample9,0.145,47,6,0


gemma4:e4b


,f1,split apart,wrong label,fused block
sample,,,,
sample1,1.000,0,0,0
sample2,0.582,8,6,0
sample3,0.929,0,1,0
sample4,1.000,0,0,0
sample5,0.385,26,3,0
sample6,0.240,5,3,3
sample7,0.500,0,1,0
sample8,0.538,0,6,0
sample9,0.957,1,0,0


## 6. What this means

**The extractor is not the problem.** It reads every document correctly; it only cuts
the text into more pieces than the annotation has items. A model that labels those
pieces consistently gets them rejoined for free and loses nothing.

**The `fused block` column is the extractor's genuine failure**, and it is small. It
is concentrated in the document whose headings are underlined rather than bold — an
underline is a drawn line, not a font attribute, so the reader cannot see it and the
heading stays glued to its answer.

**Everything else is the model.** Line-level input is simply a harder test: it asks
the model to stay consistent across a line break, and that is where capability shows.

The practical consequence: line merging, which was removed before this run, was doing
for the weaker model what a stronger model does for itself. Judge an extractor change
on the best model available, or the result will mostly measure the weakest one.
